In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score

from models.youverse.model_insightface import load_insightface_ir50_model
from models.face_recognition_model import FaceRecognitionModel
from datasets.resilience_dataset import create_dataloaders_with_partition
from datasets.face_pair_dataset import FacePairDataset


In [2]:
def get_embedding(model, image):
    model.eval()
    with torch.no_grad():
        embedding = model(image)
        embedding = torch.nn.functional.normalize(embedding)
    return embedding

model = load_insightface_ir50_model("models_bin/backbone_ir50_ms1m_epoch120.pth", device="cpu")

In [5]:
dataset = FacePairDataset("AdvLFW/images")
val_loader = DataLoader(dataset, batch_size=32, shuffle=False)

In [6]:
all_labels = []
all_scores = []

model.eval()

for batch in val_loader:
    img1 = batch['image1']
    img2 = batch['image2']
    labels = batch['label']

    emb1 = get_embedding(model, img1)
    emb2 = get_embedding(model, img2)

    sim_scores = model.cosine_similarity(emb1, emb2) 

    all_scores.extend(sim_scores.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

KeyboardInterrupt: 

In [7]:
len(all_labels)

5152

In [19]:
threshold = 0.28
preds = [1 if score > threshold else 0 for score in all_scores]

acc = accuracy_score(all_labels, preds)
roc_auc = roc_auc_score(all_labels, all_scores)

print(f"Accuracy: {acc:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

Accuracy: 0.4950
ROC AUC: 0.4500


In [12]:
base_path = 'AdvCelebA'
identity_file = os.path.join(base_path, 'identity_CelebA.txt')
partition_file = os.path.join(base_path, 'list_eval_partition_no_overlap.txt')
attack_status_file = os.path.join(base_path, 'attack_CelebA.txt')
attack_info_file = os.path.join(base_path, 'final_attack_attackid_cw.txt')
image_dir = os.path.join(base_path, 'images')

train_loader, val_loader, test_loader, dataset = create_dataloaders_with_partition(
        identity_file=identity_file,
        partition_file=partition_file,
        attack_status_file=attack_status_file,
        attack_info_file=attack_info_file,
        image_dir=image_dir,
        batch_size=32
)

print(f"Number of identities: {dataset.num_identities}")
print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Testing samples: {len(test_loader.dataset)}")

Number of identities: 10177
Training samples: 81271
Validation samples: 10132
Testing samples: 10005


In [25]:
def accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)
    correct = (predicted == labels).sum().item()
    return correct / labels.size(0)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            labels = batch['identity'].to(device)

            logits, _ = model(images)
            print(logits.shape, labels)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            total_acc += accuracy(logits, labels)

    return total_loss / len(dataloader), total_acc / len(dataloader)

In [ ]:
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
val_loss, val_acc = evaluate(model, val_loader, criterion, device)
print(f"📈 Validation accuracy: {val_loss:.4f}")

torch.Size([32, 512]) torch.Size([])


IndexError: Target 513 is out of bounds.

In [22]:
my_model = FaceRecognitionModel(num_classes=dataset.num_identities).to(device)

In [26]:
val_loss, val_acc = evaluate(my_model, val_loader, criterion, device)
print(f"📈 Validation accuracy: {val_loss:.4f}")

torch.Size([32, 10177]) tensor([ 34, 163, 297, 390, 513, 593, 675, 163, 955, 400, 739, 568, 297, 513,
        675, 955, 400, 513, 513, 593, 390, 739, 675,  34, 163, 520, 163, 675,
        163,  34, 163, 955])
torch.Size([32, 10177]) tensor([390, 390, 593, 400, 513, 955, 520, 390, 955, 297, 400, 297, 520, 739,
        593, 513, 568, 400, 163,  34, 955, 675, 520, 520, 593,  34, 739, 390,
        739, 593, 739, 675])
torch.Size([32, 10177]) tensor([675, 400, 297, 593, 513, 513, 297,  34,  34, 297, 675, 675, 400, 297,
        675, 390, 568, 297, 955, 675, 593, 568, 400, 163, 163, 390, 163, 739,
        955, 513, 163, 520])
torch.Size([32, 10177]) tensor([520,  34, 568, 513, 163, 400, 390, 513, 390, 520, 400, 163, 390, 400,
        568, 163, 297, 400, 163,  34, 568, 390, 163, 568, 163, 955, 739, 593,
         34, 675, 593, 400])
torch.Size([32, 10177]) tensor([ 400,  297,  297,  520, 8197, 8198, 8199, 8200, 8201, 8203, 8208, 8210,
        8212, 8215, 8216, 8217, 8218, 8219, 8220, 8222, 8223

KeyboardInterrupt: 